In [5]:
%matplotlib inline
%load_ext autoreload
%autoreload 2
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import math
from math import sqrt
import ROOT
import ctypes
try:
#     plt.style.use('belle2')
    # plt.style.use('belle2_serif')
    plt.style.use('belle2_modern')
except OSError:
    print("Please install belle2 matplotlib style") 
px = 1/plt.rcParams['figure.dpi']

from main.data_tools.extract_ntuples import get_pd, get_np
from main.draw_tools.decorations import b2helix, watermark
from main.draw_tools.stacking_with_error_bars import MC_stack_plot, MC_stack_plot_density

from main.data_tools.error_bars import make_data_weight
from main.data_tools.query_dataframes import cut_dfs_7types

from matplotlib.ticker import ScalarFormatter


Welcome to JupyROOT 6.26/04


In [6]:
def cal_Br_Central_value(Nsig, Nref, eff_sig, eff_ref, Br_ref_dec):
    print("Input values:")
    print("Nsig:", Nsig)
    print("Nref:", Nref)
    print("eff_sig:", eff_sig)
    print("eff_ref:", eff_ref)
    print("Br_ref_dec:", Br_ref_dec)
    
    N_real_sig = Nsig / eff_sig
    N_real_ref = Nref / eff_ref
    
    Br = Br_ref_dec * (N_real_sig / N_real_ref)
    
    return Br
    
from math import sqrt

def cal_Br_error_stat(Nsig_err, Nsig, Nref_err, Nref, central_value):
    Variance = (Nsig_err / Nsig)**2 + (Nref_err / Nref)**2
    TOTAL = sqrt(Variance) * central_value
    return TOTAL

def cal_Br_error_with_eff(Nsig_err, Nsig, Nref_err, Nref, eff_sig_err, eff_sig, eff_ref_err, eff_ref, central_value):
    Variance = (Nsig_err / Nsig)**2 + (eff_sig_err / eff_sig)**2 + (Nref_err / Nref)**2 + (eff_ref_err / eff_ref)**2
    TOTAL = sqrt(Variance) * central_value
    return TOTAL


In [7]:
from math import sqrt

# Function to calculate the central branching ratio value
def calculate_br(Nsig, Nref, eff_sig, eff_ref, Br_ref_dec):
    N_real_sig = Nsig / eff_sig
    N_real_ref = Nref / eff_ref
    Br = Br_ref_dec * (N_real_sig / N_real_ref)
    return Br
    
def calculate_br_ratio(Nsig,Nsig_err,  Nref,Nref_err, eff_sig, eff_ref):
    N_real_sig = Nsig / eff_sig
    N_real_ref = Nref / eff_ref
    Br_ratio = (N_real_sig / N_real_ref)
    Br_ratio_err = (eff_ref/eff_sig) * math.sqrt( (Nsig_err / Nref)**2 + (Nsig / (Nref**2) * Nref_err)**2 )
    print(f'Br_ratio value = {Br_ratio:.4e}')
    print(f'Br_ratio Statistical uncertainty = {Br_ratio_err:.4e}')
    return Br_ratio, Br_ratio_err

# Function to calculate statistical uncertainty
def calculate_stat_uncertainty(Nsig_err, Nsig, Nref_err, Nref, central_value):
    variance = (Nsig_err / Nsig)**2 + (Nref_err / Nref)**2
    return sqrt(variance) * central_value

# Function to print results in a formatted way
def print_results(label, central_value, stat_unc, Br_sig_dec):
    pull = (central_value - Br_sig_dec) / stat_unc
    print(f'{label} Central value = {central_value:.4e}')
    print(f'{label} Statistical uncertainty = {stat_unc:.4e}')
    print(f'{label} Pull = {pull:.4f}')
    print(f'{label} stas. unc./Central value = {stat_unc/central_value:.4e}\n')

# Error-weighted combination function
def combine_error_weighted(x, y, x_err, y_err):
    central_value = (x / x_err**2 + y / y_err**2) / (1 / x_err**2 + 1 / y_err**2)
    error = 1 / sqrt(1 / x_err**2 + 1 / y_err**2)
    return central_value, error

In [12]:
# signal_eff_error = math.sqrt(signal_eff * (1 - signal_eff) / N_gen)

def calculate_sig_eff_err(eff, N_gen):

    error = math.sqrt(eff * (1 - eff) / N_gen)
    return error

# Br(Ds+ -> eta K+)

In [18]:
Br_ref_pdg_2026 = 1.686 * 10**(-2)
Br_sig_pdg_2026 = 1.76 * 10**(-3)
Br_sig_PDG_err = 0.08 * 10**(-3)

In [19]:
Br_sig_pdg_2026/Br_ref_pdg_2026

0.10438908659549229

In [20]:
#Belle
eff_sig_cal =  0.0742
eff_ref_cal =  0.1084

In [21]:
eff_sig_err_cal = calculate_sig_eff_err(eff_sig_cal, 6e+6)

eff_sig_err_cal


eff_ref_err_cal = calculate_sig_eff_err(eff_ref_cal, 6e+6)
eff_ref_err_cal

print(f"signal eff error: {eff_sig_err_cal:.6e}, ref eff error: {eff_ref_err_cal:.6e}")

signal eff error: 1.070003e-04, ref eff error: 1.269182e-04


In [22]:
# First calculation for mode: eta -> gg
Nsig_err, Nsig, Nref_err, Nref = 429, 10716 , 1173 , 166696
# Nsig_err, Nsig, Nref_err, Nref = , , ,

eff_sig_err, eff_sig, eff_ref_err, eff_ref = eff_sig_err_cal, eff_sig_cal, eff_ref_err_cal, eff_ref_cal

central_value_1 = calculate_br(Nsig, Nref, eff_sig, eff_ref, Br_ref_pdg_2026)
stat_unc_1 = calculate_stat_uncertainty(Nsig_err, Nsig, Nref_err, Nref, central_value_1)
print_results("Mode: eta -> gg", central_value_1, stat_unc_1, Br_sig_pdg_2026)


Mode: eta -> gg Central value = 1.5834e-03
Mode: eta -> gg Statistical uncertainty = 6.4361e-05
Mode: eta -> gg Pull = -2.7439
Mode: eta -> gg stas. unc./Central value = 4.0647e-02



In [23]:
ratio_central_Br_gg, ratio_err_Br_gg  = calculate_br_ratio(Nsig,Nsig_err, Nref, Nref_err, eff_sig, eff_ref)


Br_ratio value = 9.3915e-02
Br_ratio Statistical uncertainty = 3.8174e-03


In [24]:
#Belle
eff_sig_cal = 0.0404
eff_ref_cal = 0.0650

In [25]:
eff_sig_err_cal = calculate_sig_eff_err(eff_sig_cal, 6e+6)

eff_ref_err_cal = calculate_sig_eff_err(eff_ref_cal, 6e+6)

print(f"signal eff error: {eff_sig_err_cal:.6e}, ref eff error: {eff_ref_err_cal:.6e}")

signal eff error: 8.038225e-05, ref eff error: 1.006438e-04


In [26]:
# Second calculation for mode: eta -> pipipi
Nsig_err, Nsig, Nref_err, Nref = 121, 3175, 407, 56132
# Nsig_err, Nsig, Nref_err, Nref =, , ,

eff_sig_err, eff_sig, eff_ref_err, eff_ref = eff_sig_err_cal, eff_sig_cal ,eff_ref_err_cal,eff_ref_cal

central_value_2 = calculate_br(Nsig, Nref, eff_sig, eff_ref, Br_ref_pdg_2026)
stat_unc_2 = calculate_stat_uncertainty(Nsig_err, Nsig, Nref_err, Nref, central_value_2)
print_results("Mode: eta -> pipipi", central_value_2, stat_unc_2, Br_sig_pdg_2026)

Mode: eta -> pipipi Central value = 1.5343e-03
Mode: eta -> pipipi Statistical uncertainty = 5.9523e-05
Mode: eta -> pipipi Pull = -3.7911
Mode: eta -> pipipi stas. unc./Central value = 3.8794e-02



In [27]:
ratio_central_Br_3pi, ratio_err_Br_3pi  = calculate_br_ratio(Nsig,Nsig_err, Nref, Nref_err, eff_sig, eff_ref)


Br_ratio value = 9.1005e-02
Br_ratio Statistical uncertainty = 3.5304e-03


In [28]:
# Combined error-weighted result of absoulte Br
combined_central_value, combined_error = combine_error_weighted(central_value_1, central_value_2, stat_unc_1, stat_unc_2)
print_results("Combined", combined_central_value, combined_error, Br_sig_pdg_2026)

Combined Central value = 1.5570e-03
Combined Statistical uncertainty = 4.3700e-05
Combined Pull = -4.6463
Combined stas. unc./Central value = 2.8067e-02



In [29]:
# Combined error-weighted result of Br ratio
combined_central_value, combined_error = combine_error_weighted(ratio_central_Br_gg,ratio_central_Br_3pi, ratio_err_Br_gg, ratio_err_Br_3pi)
print_results("Combined", combined_central_value, combined_error, Br_sig_pdg_2026/Br_ref_pdg_2026)

Combined Central value = 9.2346e-02
Combined Statistical uncertainty = 2.5919e-03
Combined Pull = -4.6463
Combined stas. unc./Central value = 2.8067e-02



In [ ]:
# 2.895 \pm 0.209

# Br(Ds+ -> eta K+)

In [13]:
Br_ref_pdg_2026 = 1.686 * 10**(-2)
Br_sig_pdg_2026 = 1.76 * 10**(-3)
Br_sig_PDG_err = 0.08 * 10**(-3)

In [14]:
Br_sig_pdg_2026/Br_ref_pdg_2026

0.10438908659549229

In [15]:
#CLEO2010
eff_sig_cal =  0.1140
eff_ref_cal =  0.1270

In [16]:
eff_sig_err_cal = calculate_sig_eff_err(eff_sig_cal, 6e+6)

eff_sig_err_cal


eff_ref_err_cal = calculate_sig_eff_err(eff_ref_cal, 6e+6)
eff_ref_err_cal

print(f"signal eff error: {eff_sig_err_cal:.6e}, ref eff error: {eff_ref_err_cal:.6e}")

signal eff error: 1.297459e-04, ref eff error: 1.359356e-04


In [17]:
# First calculation for mode: eta -> gg
Nsig_err, Nsig, Nref_err, Nref = 421, 222 ,  89 , 2587
# Nsig_err, Nsig, Nref_err, Nref = , , ,

eff_sig_err, eff_sig, eff_ref_err, eff_ref = eff_sig_err_cal, eff_sig_cal, eff_ref_err_cal, eff_ref_cal

central_value_1 = calculate_br(Nsig, Nref, eff_sig, eff_ref, Br_ref_pdg_2026)
stat_unc_1 = calculate_stat_uncertainty(Nsig_err, Nsig, Nref_err, Nref, central_value_1)
print_results("Mode: eta -> gg", central_value_1, stat_unc_1, Br_sig_pdg_2026)


Mode: eta -> gg Central value = 1.6118e-03
Mode: eta -> gg Statistical uncertainty = 3.0571e-03
Mode: eta -> gg Pull = -0.0485
Mode: eta -> gg stas. unc./Central value = 1.8967e+00



In [18]:
ratio_central_Br_gg, ratio_err_Br_gg  = calculate_br_ratio(Nsig,Nsig_err, Nref, Nref_err, eff_sig, eff_ref)


Br_ratio value = 9.5599e-02
Br_ratio Statistical uncertainty = 1.8132e-01
